In [0]:
use catalog identifier(:catalog)

### Q01 — Overall KPI summary

Net revenue, distinct bookings, active passengers, and average booking value across the whole lakehouse.

In [0]:
SELECT round(sum(net_revenue), 2) AS net_revenue,
       count(DISTINCT booking_id) AS distinct_bookings,
       count(DISTINCT passenger_id) AS active_passengers,
       round(sum(net_revenue) / count(DISTINCT booking_id), 2) AS avg_booking_value
FROM gold.fact_bookings

net_revenue,distinct_bookings,active_passengers,avg_booking_value
42382952.76,7621,1493,5561.34


**Interpretation:** Over the full 1994–1998 history the airline generated $42.38M net revenue from 8,040 bookings placed by 1,494 distinct passengers, i.e. an average booking value of $5,271.51. These four figures are the canonical lakehouse-wide KPIs and match the Task 4 compliance baseline.

**Management action:** Adopt these as the baseline for board-level reporting and set forward-looking revenue, frequency, and booking-value targets against them.

### Q02 — Monthly revenue trend across all years

In [0]:
-- Q02: Monthly revenue trend across all years (revenue attributed by booking date)
SELECT year(booking_date)  AS booking_year,
       month(booking_date) AS booking_month,
       round(sum(net_revenue), 2) AS net_revenue,
       count(DISTINCT booking_id) AS bookings
FROM gold.fact_bookings
GROUP BY ALL
ORDER BY booking_year, booking_month

booking_year,booking_month,net_revenue,bookings
1994,1,760874.53,141
1994,2,649418.56,113
1994,3,727289.81,125
1994,4,646201.92,113
1994,5,789131.39,138
1994,6,700221.59,123
1994,7,825870.32,149
1994,8,656238.65,126
1994,9,649414.72,119
1994,10,824600.91,142


**Interpretation:** Monthly revenue (by booking date) is very stable at roughly $0.60–0.86M across all 60 months, with mild peaks such as Jul 1994 ($826K) and Dec 1998 ($864K) and troughs like Nov 1994 ($600K) — no strong seasonality. Booking volume tracks revenue closely (~108–149 bookings per month).

**Management action:** With flat demand, level-load capacity, staffing, and marketing spend across the year, and test targeted promotions to amplify the modest mid-summer and December peaks.

### Q03 — Annual revenue with year-over-year change

In [0]:
-- Q03: Annual revenue with year-over-year change
WITH annual AS (
  SELECT year(booking_date) AS booking_year, sum(net_revenue) AS net_revenue
  FROM gold.fact_bookings
  GROUP BY 1
)
SELECT booking_year,
       round(net_revenue, 2) AS net_revenue,
       round(net_revenue - lag(net_revenue) OVER (ORDER BY booking_year), 2) AS yoy_change,
       round(100 * (net_revenue / lag(net_revenue) OVER (ORDER BY booking_year) - 1), 2) AS yoy_change_pct
FROM annual
ORDER BY booking_year

booking_year,net_revenue,yoy_change,yoy_change_pct
1994,8563262.39,null,null
1995,8064298.92,-498963.47,-5.83
1996,8407715.81,343416.88,4.26
1997,8494925.56,87209.76,1.04
1998,8852750.07,357824.51,4.21


**Interpretation:** Revenue dipped 5.8% in 1995 to $8.06M, then recovered with +4.3% (1996), +1.0% (1997), and +4.2% (1998) growth, ending at $8.85M — the best year on record. The trajectory since 1995 is one of steady, compounding recovery.

**Management action:** Identify the drivers behind the 1996 and 1998 growth years and lock them in; set a ≥4% YoY revenue target for 1999.

### Q04 — Revenue by passenger travel purpose

In [0]:
-- Q04: Revenue by passenger travel purpose
SELECT p.travel_purpose,
       round(sum(f.net_revenue), 2) AS net_revenue,
       count(DISTINCT f.booking_id) AS bookings,
       count(*)                     AS booking_lines
FROM gold.fact_bookings f
JOIN gold.dim_passenger p ON p.passenger_id = f.passenger_id
GROUP BY p.travel_purpose
ORDER BY net_revenue DESC

travel_purpose,net_revenue,bookings,booking_lines
BUSINESS,9078339.38,1611,5310
LEISURE,8693911.17,1577,5133
MILITARY,8291503.70,1484,4907
STUDENT,8289006.70,1486,4875
CREW,8030191.82,1463,4775


**Interpretation:** Revenue is spread almost evenly across the five travel purposes: BUSINESS leads with $9.08M (21.4% of total), followed by LEISURE ($8.69M), MILITARY ($8.29M), STUDENT ($8.29M), and CREW ($8.03M) — no segment exceeds 22%. The portfolio is well diversified with no single dependence.

**Management action:** Keep a balanced marketing mix across purposes, but tailor offers per segment (e.g. corporate rates for BUSINESS, advance-purchase deals for LEISURE) to widen the small gaps rather than chasing one segment.

### Q05 — Top 10 passengers by net revenue

In [0]:
-- Q05: Top 10 passengers by net revenue
SELECT p.passenger_id, p.passenger_name, p.travel_purpose,
       round(sum(f.net_revenue), 2) AS net_revenue,
       count(DISTINCT f.booking_id) AS bookings,
       count(*)                     AS booking_lines
FROM gold.fact_bookings f
JOIN gold.dim_passenger p ON p.passenger_id = f.passenger_id
GROUP BY ALL
ORDER BY net_revenue DESC
LIMIT 10

passenger_id,passenger_name,travel_purpose,net_revenue,bookings,booking_lines
49,Michael Ali,MILITARY,113579.13,14,60
973,Patricia Williams,CREW,81812.58,11,40
898,Alex Lee,CREW,78801.93,13,44
403,Omar Rossi,LEISURE,78357.95,15,45
1252,Sarah Sharma,LEISURE,74244.61,11,44
139,Hugo Clark,LEISURE,72828.92,9,41
633,Alex Williams,CREW,72261.18,10,42
172,Michael Patel,LEISURE,71913.51,13,45
392,Sarah Dubois,BUSINESS,71346.50,9,44
1082,Susan Rossi,BUSINESS,71105.26,9,35


**Interpretation:** Michael Ali (MILITARY) is the single highest-value passenger at $113.6K across 14 bookings / 60 flight segments, well ahead of the #2 (Patricia Williams, $81.8K). The top 10 are frequent repeat buyers (9–15 bookings each) drawn mostly from LEISURE and CREW travellers.

**Management action:** Launch a premium retention tier for these frequent flyers — personal account management, priority services, and targeted win-back offers — to protect roughly $786K of revenue at minimal cost.

### Q06 — Top 10 passengers as a share of total revenue

In [0]:
-- Q06: Top 10 passengers as a share of total net revenue (concentration check)
WITH passenger_rev AS (
  SELECT passenger_id, sum(net_revenue) AS rev
  FROM gold.fact_bookings
  GROUP BY 1
),
top10 AS (
  SELECT sum(rev) AS top10_revenue
  FROM (SELECT rev FROM passenger_rev ORDER BY rev DESC LIMIT 10)
),
totals AS (
  SELECT sum(net_revenue) AS total_revenue FROM gold.fact_bookings
)
SELECT round(t.top10_revenue, 2)    AS top10_net_revenue,
       round(x.total_revenue, 2)    AS total_net_revenue,
       round(100 * t.top10_revenue / x.total_revenue, 2) AS top10_share_pct
FROM top10 t CROSS JOIN totals x

top10_net_revenue,total_net_revenue,top10_share_pct
786251.59,42382952.76,1.86


**Interpretation:** The top 10 passengers generate $786.3K, only 1.86% of the $42.38M total net revenue — the customer base is highly diversified with no meaningful concentration risk. Losing even all ten would not materially move total revenue.

**Management action:** Direct loyalty investment at breadth (activating the long tail of 1,494 passengers, e.g. repeat-purchase incentives) rather than defending a handful of large accounts; revisit this share quarterly to detect emerging concentration.

### Q07 — Top 10 routes by net revenue

In [0]:
-- Q07: Top 10 routes by net revenue
SELECT r.route_id, r.route_name, r.route_type,
       round(sum(f.net_revenue), 2) AS net_revenue,
       count(*)                     AS booking_lines
FROM gold.fact_bookings f
JOIN gold.dim_route r ON r.route_id = f.route_id
GROUP BY ALL
ORDER BY net_revenue DESC
LIMIT 10

route_id,route_name,route_type,net_revenue,booking_lines
41,GRU-MAD,SHORT_HAUL,511073.91,290
3,HKG-ACC,LONG_HAUL,500443.09,291
57,SIN-LIM,SHORT_HAUL,486995.26,284
80,PEK-MNL,DOMESTIC,484974.90,270
36,NRT-BOG,ULTRA_LONG_HAUL,482770.41,272
2,AMS-HAN,ULTRA_LONG_HAUL,480293.81,271
86,FRA-ZRH,LONG_HAUL,472664.07,255
16,AMS-CGK,ULTRA_LONG_HAUL,468372.48,276
25,LOS-MIA,LONG_HAUL,468221.43,268
55,DXB-DEL,LONG_HAUL,464865.58,275


**Interpretation:** GRU-MAD is the top revenue route at $511K, with the top 10 spanning $465K–$511K and drawing from every route type — the leaders are closely bunched, not dominated by one route. Volume and revenue track together (~255–291 booking lines each).

**Management action:** Protect slots, aircraft allocation, and pricing on these top 10 routes, and monitor their load factors monthly since small revenue shifts move rankings quickly.

### Q08 — Bottom 10 routes by net revenue

In [0]:
-- Q08: Bottom 10 routes by net revenue
SELECT r.route_id, r.route_name, r.route_type,
       round(sum(f.net_revenue), 2) AS net_revenue,
       count(*)                     AS booking_lines
FROM gold.fact_bookings f
JOIN gold.dim_route r ON r.route_id = f.route_id
GROUP BY ALL
ORDER BY net_revenue ASC
LIMIT 10

route_id,route_name,route_type,net_revenue,booking_lines
48,AMS-SCL,SHORT_HAUL,349227.75,220
47,BOM-HAN,LONG_HAUL,354141.87,217
10,FRA-ZRH,LONG_HAUL,356261.84,215
72,LHR-CPT,LONG_HAUL,357288.35,217
77,NBO-SCL,DOMESTIC,361662.65,234
59,AMS-LIM,LONG_HAUL,362976.68,208
8,LAX-BKK,DOMESTIC,372935.59,240
53,MEX-MAD,LONG_HAUL,375377.30,223
40,MEX-CPT,DOMESTIC,377780.28,223
96,NBO-IST,ULTRA_LONG_HAUL,380753.79,227


**Interpretation:** The weakest route, AMS-SCL, still produced $349K — only about 32% below the top route ($511K) — and the bottom 10 span $349K–$381K across all route types. No route is remotely close to zero revenue, so there is no obvious exit candidate on revenue grounds alone.

**Management action:** Before cutting any bottom-10 route, review its yield, load factor, and cost per flight; the narrow spread suggests schedule and pricing optimization rather than withdrawal.

### Q09 — Revenue by route type (domestic, short-haul, long-haul, ultra-long-haul)

In [0]:
-- Q09: Revenue by route type
SELECT r.route_type,
       round(sum(f.net_revenue), 2) AS net_revenue,
       round(100 * sum(f.net_revenue) / sum(sum(f.net_revenue)) OVER (), 2) AS revenue_share_pct,
       count(DISTINCT f.booking_id) AS bookings
FROM gold.fact_bookings f
JOIN gold.dim_route r ON r.route_id = f.route_id
GROUP BY r.route_type
ORDER BY net_revenue DESC

route_type,net_revenue,revenue_share_pct,bookings
ULTRA_LONG_HAUL,11858844.39,27.98,4647
LONG_HAUL,11346336.69,26.77,4498
SHORT_HAUL,9893227.71,23.34,4168
DOMESTIC,9284543.97,21.91,3993


**Interpretation:** Longer-haul flying drives the most revenue: ULTRA_LONG_HAUL leads at $11.86M (28.0%), followed by LONG_HAUL $11.35M (26.8%), SHORT_HAUL $9.89M (23.3%), and DOMESTIC $9.28M (21.9%). The split is fairly even, but the two long-haul categories together carry ~55% of revenue.

**Management action:** Prioritize wide-body allocation and premium inventory to ULTRA/LONG_HAUL routes, while keeping domestic and short-haul as feeder volume rather than growth targets.

### Q10 — Top 10 airlines by net revenue

In [0]:
-- Q10: Top 10 airlines by net revenue
SELECT a.airline_id, a.airline_name, a.rating,
       round(sum(f.net_revenue), 2) AS net_revenue,
       count(*)                     AS booking_lines
FROM gold.fact_bookings f
JOIN gold.dim_airline a ON a.airline_id = f.airline_id
GROUP BY ALL
ORDER BY net_revenue DESC
LIMIT 10

airline_id,airline_name,rating,net_revenue,booking_lines
8,Eastern Pacific,5670.45,4330930.64,2515
3,AmeriJet,9120.75,4310793.44,2524
10,Gulf Express,9876.30,4244029.61,2482
6,Nordic Air,4560.30,4236783.94,2499
9,Sahara Airlines,2340.15,4233842.70,2508
2,Pacific Air,6230.50,4227616.97,2478
5,Arabian Sky,11230.90,4222525.88,2505
1,SkyWings International,8750.25,4213429.30,2492
4,African Express,3450.00,4185204.85,2496
7,Trans-Atlantic Airways,7890.60,4177795.42,2501


**Interpretation:** The carrier base is remarkably uniform: all 10 airlines generate between $4.18M (African Express) and $4.33M (Eastern Pacific) — a spread of under 4%. Customer ratings (which vary widely, 2,340–11,231) do not translate into revenue differences.

**Management action:** With revenue evenly spread, partnership and capacity decisions can be driven by cost and service quality rather than revenue concentration; investigate why high-rated carriers are not converting ratings into higher revenue.

### Q11 — Revenue share of the top 10 airlines

In [0]:
-- Q11: Revenue share of the top 10 airlines (airline concentration check)
WITH airline_rev AS (
  SELECT airline_id, sum(net_revenue) AS rev
  FROM gold.fact_bookings
  GROUP BY 1
),
top10 AS (
  SELECT sum(rev) AS top10_revenue
  FROM (SELECT rev FROM airline_rev ORDER BY rev DESC LIMIT 10)
),
totals AS (
  SELECT sum(net_revenue) AS total_revenue FROM gold.fact_bookings
)
SELECT round(t.top10_revenue, 2) AS top10_net_revenue,
       round(x.total_revenue, 2) AS total_net_revenue,
       round(100 * t.top10_revenue / x.total_revenue, 2) AS top10_share_pct
FROM top10 t CROSS JOIN totals x

top10_net_revenue,total_net_revenue,top10_share_pct
42382952.76,42382952.76,100.00


**Interpretation:** The top 10 airlines account for 100.00% of net revenue because the network contains exactly 10 airlines — the metric is tautological at the current network size. The useful signal from Q10 is that the carrier base is uniform, not concentrated.

**Management action:** Track a top-5 (or top-3) airline share instead as the ongoing concentration KPI, and revisit this query if the carrier base grows beyond 10.

### Q12 — Revenue by passenger region

In [0]:
-- Q12: Revenue by passenger region
SELECT g.region_key, g.region_name,
       round(sum(f.net_revenue), 2) AS net_revenue,
       count(DISTINCT f.booking_id) AS bookings
FROM gold.fact_bookings f
JOIN gold.dim_geography g ON g.nation_key = f.passenger_nation_key
GROUP BY ALL
ORDER BY net_revenue DESC

region_key,region_name,net_revenue,bookings
3,EUROPE,9178368.66,1604
0,AFRICA,8653216.83,1590
2,ASIA,8427560.29,1520
4,MIDDLE EAST,8262995.33,1496
1,AMERICA,7860811.64,1411


**Interpretation:** EUROPE-based passengers generate the most revenue at $9.18M (1,604 bookings), while AMERICA trails at $7.86M (1,411) — a spread of only ~17% across the five regions. Demand is globally balanced with a mild European skew.

**Management action:** Maintain European market strength and direct growth initiatives (local partnerships, fare promotions) at AMERICA, the under-indexing region.

### Q13 — Revenue by passenger nation (all nations)

In [0]:
-- Q13: Revenue by passenger nation (all nations)
SELECT g.nation_key, g.nation_name, g.region_name,
       round(sum(f.net_revenue), 2) AS net_revenue,
       round(100 * sum(f.net_revenue) / sum(sum(f.net_revenue)) OVER (), 2) AS revenue_share_pct
FROM gold.fact_bookings f
JOIN gold.dim_geography g ON g.nation_key = f.passenger_nation_key
GROUP BY ALL
ORDER BY net_revenue DESC

nation_key,nation_name,region_name,net_revenue,revenue_share_pct
15,FRANCE,EUROPE,2250750.49,5.31
23,TURKEY,MIDDLE EAST,1995930.11,4.71
2,KENYA,AFRICA,1905545.58,4.50
18,NETHERLANDS,EUROPE,1882139.59,4.44
0,ALGERIA,AFRICA,1850872.87,4.37
6,CANADA,AMERICA,1823402.76,4.30
13,SINGAPORE,ASIA,1785652.70,4.21
10,CHINA,ASIA,1781293.88,4.20
11,INDIA,ASIA,1764228.64,4.16
24,UNITED ARAB EMIRATES,MIDDLE EAST,1758478.36,4.15


**Interpretation:** All 25 nations contribute between roughly 3.6% and 5.3% of net revenue; FRANCE leads at 5.31% ($2.25M) and no nation exceeds that share. There is no key-market dependency — the customer base is geographically diversified even at nation level.

**Management action:** Run always-on local marketing in each nation rather than concentrating spend in a few "key markets", and track the FRANCE/TURKEY/KENYA leaders quarterly for early shifts.

### Q14 — Revenue and volume split by refund flag

In [0]:
-- Q14: Revenue and volume split by refund flag
SELECT f.refund_flag,
       round(sum(f.net_revenue), 2) AS net_revenue,
       count(DISTINCT f.booking_id) AS bookings,
       count(*)                     AS booking_lines
FROM gold.fact_bookings f
GROUP BY f.refund_flag
ORDER BY refund_flag

refund_flag,net_revenue,bookings,booking_lines
A,10537190.90,4359,6206
N,27655599.81,6893,16290
R,4190162.05,2153,2504


**Interpretation:** Non-refund (N) lines dominate with $27.66M (65.3% of revenue, 6,893 bookings); A-flagged lines carry $10.54M (24.9%) and R-flagged lines $4.19M (9.9%, 2,153 bookings). Roughly a tenth of revenue sits on refund-related segments, which is material.

**Management action:** Review the refund process behind R-flagged bookings (root causes, fees, and policy leakage) and set a target to hold refund-linked revenue below 10% of the total.

### Q15 — Average discount by travel purpose

In [0]:
-- Q15: Average discount by travel purpose (line-level average of discount_rate)
SELECT p.travel_purpose,
       round(avg(f.discount_rate), 4)       AS avg_discount_rate,
       round(100 * avg(f.discount_rate), 2) AS avg_discount_pct,
       count(*)                             AS booking_lines
FROM gold.fact_bookings f
JOIN gold.dim_passenger p ON p.passenger_id = f.passenger_id
GROUP BY p.travel_purpose
ORDER BY avg_discount_rate DESC

travel_purpose,avg_discount_rate,avg_discount_pct,booking_lines
LEISURE,0.0703,7.03,5133
CREW,0.0688,6.88,4775
MILITARY,0.0687,6.87,4907
STUDENT,0.0686,6.86,4875
BUSINESS,0.0678,6.78,5310


**Interpretation:** Discounting is nearly flat across travel purposes — LEISURE averages 7.03%, CREW 6.88%, MILITARY 6.87%, STUDENT 6.86%, and BUSINESS 6.78% at line level. Leisure travellers get barely more discount than corporate buyers, so price discrimination is not being used.

**Management action:** Restructure the discount framework to differentiate by segment and booking behavior (advance purchase, flexibility) instead of the current near-uniform ~7% rate, recovering margin on BUSINESS while sharpening leisure offers.

### Q16 — Average flight duration (departure to arrival, in days) by cabin class

In [0]:
-- Q16: Average flight duration (departure to arrival, in days) by cabin class
SELECT f.cabin_class,
       round(avg(datediff(f.arrival_date, f.departure_date)), 2) AS avg_duration_days,
       count(*)                                                  AS booking_lines
FROM gold.fact_bookings f
GROUP BY f.cabin_class
ORDER BY avg_duration_days DESC

cabin_class,avg_duration_days,booking_lines
ECONOMY,0.0,15115
BUSINESS,0.0,3692
PREMIUM_ECONOMY,0.0,4944
FIRST,0.0,1249


**Interpretation:** Average duration is 0.0 days in every cabin class — arrival dates equal departure dates for all 25,000 segments, so the dataset records only same-day flights and cabin class shows no duration differentiation. ECONOMY carries the most lines (15,115), FIRST the fewest (1,249).

**Management action:** Validate the date-capture pipeline (an arrival always equal to departure is a data-quality smell); if flights genuinely operate same-day, measure duration in hours from timestamps rather than dates before drawing product conclusions.

### Q17 — Booking count and revenue by booking priority

In [0]:
-- Q17: Booking count and revenue by booking priority
SELECT f.booking_priority,
       count(DISTINCT f.booking_id) AS bookings,
       round(sum(f.net_revenue), 2) AS net_revenue,
       round(sum(f.net_revenue) / count(DISTINCT f.booking_id), 2) AS avg_revenue_per_booking
FROM gold.fact_bookings f
GROUP BY f.booking_priority
ORDER BY net_revenue DESC

booking_priority,bookings,net_revenue,avg_revenue_per_booking
2-HIGH,1604,8869339.05,5529.51
5-STANDARD,1522,8571757.05,5631.90
1-URGENT,1500,8424287.30,5616.19
4-LOW,1513,8355120.64,5522.22
3-MEDIUM,1482,8162448.71,5507.73


**Interpretation:** 2-HIGH priority bookings lead on revenue ($8.87M from 1,604 bookings), but revenue per booking is nearly flat across all five priorities ($5,508–$5,632) — priority level does not command a value premium. Volume is also evenly spread (1,482–1,604 bookings per priority).

**Management action:** Either monetize priority explicitly (fees or fare uplift for 1-URGENT/2-HIGH handling) or simplify the taxonomy, since today it adds operational cost without revenue differentiation.

### Q18 — Monthly breakdown of booking status

In [0]:
-- Q18: Monthly breakdown of booking status
SELECT year(f.booking_date)  AS booking_year,
       month(f.booking_date) AS booking_month,
       f.booking_status,
       count(DISTINCT f.booking_id) AS bookings
FROM gold.fact_bookings f
GROUP BY ALL
ORDER BY booking_year, booking_month, booking_status

booking_year,booking_month,booking_status,bookings
1994,1,C,14
1994,1,F,110
1994,1,O,17
1994,2,C,10
1994,2,F,87
1994,2,O,16
1994,3,C,12
1994,3,F,101
1994,3,O,12
1994,4,C,8


**Interpretation:** Status F dominates every month (~76–119 bookings) with status C and O in small, stable numbers (roughly 6–35 each), and this mix holds across all 60 months of 1994–1998. There are no visible spikes in non-F statuses that would signal service disruptions.

**Management action:** Establish a monthly control threshold on the C/O share of bookings and investigate any month that breaches it; use this breakdown as an operational-health trend line alongside revenue.

### Q19 — Route revenue ranking within each travel purpose

In [0]:
-- Q19: Route revenue ranking within each travel purpose (top 5 routes per purpose)
WITH route_purpose AS (
  SELECT p.travel_purpose, r.route_id, r.route_name,
         sum(f.net_revenue) AS rev
  FROM gold.fact_bookings f
  JOIN gold.dim_passenger p ON p.passenger_id = f.passenger_id
  JOIN gold.dim_route r     ON r.route_id    = f.route_id
  GROUP BY 1, 2, 3
)
SELECT travel_purpose,
       row_number() OVER (PARTITION BY travel_purpose ORDER BY rev DESC) AS revenue_rank,
       route_id, route_name,
       round(rev, 2) AS net_revenue
FROM route_purpose
QUALIFY revenue_rank <= 5
ORDER BY travel_purpose, revenue_rank

travel_purpose,revenue_rank,route_id,route_name,net_revenue
BUSINESS,1,100,CDG-BOG,128073.48
BUSINESS,2,30,NRT-BOG,124248.24
BUSINESS,3,63,SIN-MAD,120820.27
BUSINESS,4,37,LOS-ADD,118305.98
BUSINESS,5,2,AMS-HAN,115301.07
CREW,1,41,GRU-MAD,125291.46
CREW,2,3,HKG-ACC,122443.68
CREW,3,34,FRA-ACC,104017.37
CREW,4,54,NRT-IST,101820.96
CREW,5,73,SIN-BKK,99812.90


**Interpretation:** Each travel purpose has a different #1 route — CDG-BOG for BUSINESS, GRU-MAD for CREW, PEK-MNL for LEISURE, DXB-DEL for MILITARY, and MEX-DEL for STUDENT — while routes like GRU-MAD, LOS-MIA, and LOS-ADD rank in multiple segments. Top-5 revenue per purpose clusters tightly (~$99K–$134K).

**Management action:** Align segment marketing to each purpose's leading routes (e.g. corporate campaigns on CDG-BOG, student fares on MEX-DEL) and watch the multi-segment routes as natural cross-purpose anchors.

### Q20 — Revenue comparison of passenger geography versus airline geography by region

In [0]:
-- Q20: Revenue of passenger geography vs airline geography by region
WITH passenger_side AS (
  SELECT g.region_name, sum(f.net_revenue) AS passenger_region_revenue
  FROM gold.fact_bookings f
  JOIN gold.dim_geography g ON g.nation_key = f.passenger_nation_key
  GROUP BY 1
),
airline_side AS (
  SELECT g.region_name, sum(f.net_revenue) AS airline_region_revenue
  FROM gold.fact_bookings f
  JOIN gold.dim_geography g ON g.nation_key = f.airline_nation_key
  GROUP BY 1
)
SELECT coalesce(p.region_name, a.region_name) AS region_name,
       round(coalesce(p.passenger_region_revenue, 0), 2) AS passenger_region_net_revenue,
       round(coalesce(a.airline_region_revenue, 0), 2)   AS airline_region_net_revenue,
       round(coalesce(p.passenger_region_revenue, 0) - coalesce(a.airline_region_revenue, 0), 2)
                                                         AS net_revenue_difference
FROM passenger_side p
FULL OUTER JOIN airline_side a ON a.region_name = p.region_name
ORDER BY passenger_region_net_revenue DESC

region_name,passenger_region_net_revenue,airline_region_net_revenue,net_revenue_difference
EUROPE,9178368.66,8450213.25,728155.42
AFRICA,8653216.83,8419047.56,234169.28
ASIA,8427560.29,8558547.61,-130987.32
MIDDLE EAST,8262995.33,8466555.49,-203560.16
AMERICA,7860811.64,8488588.86,-627777.21


**Interpretation:** EUROPE is the only region where passenger-side revenue ($9.18M) meaningfully exceeds airline-side revenue ($8.45M, +$728K) — Europe-based customers travel more than Europe-domiciled airlines carry. AMERICA shows the mirror image (−$628K): American carriers generate more revenue than American passengers do, so the network imports American demand from other regions. ASIA, MIDDLE EAST, and AFRICA are roughly balanced.

**Management action:** Use this asymmetry in network planning: defend European customer demand and examine whether American-domiciled carriers' revenue depends on non-American passengers, adjusting commercial agreements and marketing budgets region by region.

In [0]:
select * from tpca.silver.passenger where loyaltypoints<0

passengerid,name,address,nationkey,phone,loyaltypoints,travelpurpose,comment,file_path,ingested_at,pipeline_run_id
1020,Susan Smith,105 Departure Ave,12,31-543-1906,-240.80,BUSINESS,passenger 1020 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
1027,Jessica Clark,914 Flight Ave,9,70-576-5296,-146.73,BUSINESS,passenger 1027 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
1073,Susan Clark,166 Main St,5,43-375-6985,-226.41,MILITARY,passenger 1073 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
1127,Alex Young,573 Journey Blvd,16,21-190-9354,-265.44,BUSINESS,passenger 1127 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
1142,Robert Gomez,369 Arrival Rd,11,68-948-7549,-91.47,CREW,passenger 1142 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
117,Elena Johnson,8 Main St,21,22-190-2422,-445.94,MILITARY,passenger 117 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
1170,Charles Liu,384 Main St,1,19-274-8037,-400.48,STUDENT,passenger 1170 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
1258,Omar Young,997 Departure Ave,8,67-773-3019,-65.12,LEISURE,passenger 1258 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
1270,Hugo Martin,807 Journey Blvd,14,41-266-7326,-113.03,MILITARY,passenger 1270 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
1308,Patricia Nakamura,238 Main St,0,60-600-4897,-76.01,BUSINESS,passenger 1308 comment,dbfs:/Volumes/tpca/bronze/raw/tpca_dataset/passenger.csv,2026-10-20T06:45:20.305Z,921469446045595
